In [ ]:
# 1. Установка библиотек в Colab
# !pip install -q transformers datasets peft trl bitsandbytes

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# 2. Настройка квантования (чтобы модель влезла в память бесплатного Colab)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "Qwen/Qwen2.5-0.5B-Instruct" # Очень легкая модель для тестов

# 3. Загрузка модели и токенизатора
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 4. Настройка PEFT / LoRA (указываем, что будем обучать только малую часть весов)
peft_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 5. Загрузка демо-датасета (например, диалоги службы поддержки).
# В реальности здесь будет ваш локальный CSV/JSON файл
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train[:500]") # Датасет содержит пары "вопрос-ответ" для обучения модели поддержки клиентов

def format_prompt(example):
    # Формируем текст: Вопрос -> Ответ
    text = f"<|im_start|>user\n{example['instruction']}<|im_end|>\n<|im_start|>assistant\n{example['response']}<|im_end|>"
    return {"text": text}

dataset = dataset.map(format_prompt)

# 6. Запуск обучения
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    # dataset_text_field="text",
    # max_seq_length=256,
    # peft_config=peft_config,
    args=TrainingArguments( # Желательнее использовать SFTConfig
        output_dir="./results",
        max_steps=100,           # Для демо-целей всего 100 шагов
        per_device_train_batch_size=4,
        optim="paged_adamw_8bit"
    ),
    processing_class=tokenizer
)
trainer.train()

# 7. Сохранение адаптеров и слияние с базовой моделью
# Сохраняем итоговую модель в папку (ее потом нужно скачать на локальный компьютер)
trainer.model.save_pretrained("./my-support-model")
tokenizer.save_pretrained("./my-support-model")
print("Обучение завершено. Папку ./my-support-model можно скачивать!")